In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("spambase.data", header=None)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [9]:
def k_fold_cv(X, y, model, k):

    n = len(X)

    indices = np.arange(n)
    np.random.shuffle(indices)

    folds = np.array_split(indices, k)

    errors = []

    for i in range(k):

        val_idx = folds[i]

        train_idx = np.concatenate(
            [folds[j] for j in range(k) if j != i]
        )

        X_train = X[train_idx]
        y_train = y[train_idx]

        X_val = X[val_idx]
        y_val = y[val_idx]

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        accuracy = accuracy_score(y_val, preds)

        error = 1 - accuracy

        errors.append(error)

        print("Fold", i+1, "Validation Error:", error)

    avg_error = np.mean(errors)

    print("Average Validation Error:", avg_error)

    return avg_error

In [10]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "LDA": LinearDiscriminantAnalysis()
}

k_values = [5,10]

results = []

for name, model in models.items():

    for k in k_values:

        print("\nModel:", name, "| k =", k)

        avg_error = k_fold_cv(X, y, model, k)

        results.append([name, k, avg_error])


Model: Logistic Regression | k = 5
Fold 1 Validation Error: 0.0662323561346363
Fold 2 Validation Error: 0.07499999999999996
Fold 3 Validation Error: 0.08152173913043481
Fold 4 Validation Error: 0.06847826086956521
Fold 5 Validation Error: 0.07065217391304346
Average Validation Error: 0.07237690600953595

Model: Logistic Regression | k = 10
Fold 1 Validation Error: 0.09110629067245124
Fold 2 Validation Error: 0.07826086956521738
Fold 3 Validation Error: 0.05434782608695654
Fold 4 Validation Error: 0.05869565217391304
Fold 5 Validation Error: 0.07608695652173914
Fold 6 Validation Error: 0.08478260869565213
Fold 7 Validation Error: 0.07608695652173914
Fold 8 Validation Error: 0.08260869565217388
Fold 9 Validation Error: 0.08043478260869563
Fold 10 Validation Error: 0.06304347826086953
Average Validation Error: 0.07454541167594077

Model: LDA | k = 5
Fold 1 Validation Error: 0.12486427795874055
Fold 2 Validation Error: 0.10652173913043483
Fold 3 Validation Error: 0.10978260869565215
Fold 

In [11]:
cv_results = pd.DataFrame(
    results,
    columns=["Model","k","Average Validation Error"]
)

print("\nFinal Cross Validation Results")
print(cv_results)


Final Cross Validation Results
                 Model   k  Average Validation Error
0  Logistic Regression   5                  0.072377
1  Logistic Regression  10                  0.074545
2                  LDA   5                  0.112147
3                  LDA  10                  0.114321
